In [0]:
-- Game - Box Score
-- Team totals for both teams in one game: score, shots, attempts, hits, special teams, faceoffs.
-- Goal and shot totals come from play-by-play and leave out shootouts.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    2025020001 AS game_id   -- e.g. 2025020001 (season 2025, 02 = regular season, game 0001); see Game - Find Game ID
),
team_games AS (                          -- the two teams in the chosen game (in-progress games too)
  SELECT g.game_id, g.season, g.game_date, g.last_period_type, g.is_final,
         t.team_id, t.abbrev AS team, o.abbrev AS opponent,
         t.team_id = g.home_team_id AS is_home, g.home_team_id,
         IF(t.team_id = g.home_team_id, g.home_final_score, g.away_final_score) AS final_gf,
         IF(t.team_id = g.home_team_id, g.away_final_score, g.home_final_score) AS final_ga,
         CASE WHEN NOT g.is_final THEN 'In progress'
              WHEN IF(t.team_id = g.home_team_id, g.home_final_score > g.away_final_score,
                                                  g.away_final_score > g.home_final_score) THEN 'W'
              WHEN g.game_type = 2 AND g.last_period_type IN ('OT', 'SO') THEN 'OTL'
              ELSE 'L' END AS result
  FROM nhl.gold.dim_game g
  JOIN nhl.gold.dim_team t ON t.team_id IN (g.home_team_id, g.away_team_id)
  JOIN nhl.gold.dim_team o ON o.team_id = IF(t.team_id = g.home_team_id, g.away_team_id, g.home_team_id)
  CROSS JOIN params pr
  WHERE g.game_id = pr.game_id
),
team_events AS (                         -- plays in the game, from each team's side
  SELECT tg.team, tg.team_id, tg.opponent, tg.is_home,
         IF(f.event_owner_team_id = tg.team_id, 'for', 'against') AS side,
         IF(f.event_owner_team_id = tg.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength,
         br.team_id AS blocker_team_id,
         f.*
  FROM nhl.gold.fact_event f
  JOIN team_games tg ON f.game_id = tg.game_id
  LEFT JOIN nhl.silver.game_rosters br                -- blocked shots belong to the shooter's team,
         ON br.game_id = f.game_id AND br.player_id = f.blocking_player_id   -- so credit blocks by roster
  WHERE COALESCE(f.period_type, '') <> 'SO'    -- shootout attempts aren't stats
),
game_stats AS (                          -- one row per team
  SELECT tg.team, tg.opponent, tg.is_home, tg.result, tg.last_period_type, tg.final_gf, tg.final_ga,
         COUNT_IF(te.type_desc_key = 'goal' AND te.side = 'for') AS gf,
         COUNT_IF(te.type_desc_key IN ('goal', 'shot-on-goal') AND te.side = 'for') AS sf,
         COUNT_IF(te.type_desc_key IN ('goal', 'shot-on-goal', 'missed-shot', 'blocked-shot') AND te.side = 'for') AS cf,
         COUNT_IF(te.type_desc_key IN ('goal', 'shot-on-goal', 'missed-shot', 'blocked-shot') AND te.side = 'against') AS ca,
         COUNT_IF(te.type_desc_key = 'goal' AND te.side = 'for' AND te.strength = 'PP') AS pp_gf,
         COUNT_IF(te.type_desc_key = 'goal' AND te.side = 'for' AND te.strength = 'SH') AS sh_gf,
         COUNT_IF(te.type_desc_key = 'penalty' AND te.side = 'against'
               AND te.penalty_type_code IN ('MIN', 'BEN')) AS minors_drawn,
         SUM(IF(te.type_desc_key = 'penalty' AND te.side = 'for', COALESCE(te.penalty_duration, 0), 0)) AS pim,
         COUNT_IF(te.type_desc_key = 'hit' AND te.side = 'for') AS hits_for,
         COUNT_IF(te.type_desc_key = 'blocked-shot' AND te.blocker_team_id = te.team_id) AS blocks_for,
         COUNT_IF(te.type_desc_key = 'takeaway' AND te.side = 'for') AS takeaways,
         COUNT_IF(te.type_desc_key = 'giveaway' AND te.side = 'for') AS giveaways,
         COUNT_IF(te.type_desc_key = 'faceoff' AND te.side = 'for') AS fo_wins,
         COUNT_IF(te.type_desc_key = 'faceoff' AND te.side = 'against') AS fo_losses
  FROM team_games tg
  LEFT JOIN team_events te ON te.game_id = tg.game_id AND te.team_id = tg.team_id
  GROUP BY tg.team, tg.opponent, tg.is_home, tg.result, tg.last_period_type, tg.final_gf, tg.final_ga
)
SELECT team,
       IF(is_home, 'Home', 'Away') AS venue,
       final_gf AS score,
       result,
       last_period_type AS decided_in,
       sf AS shots,
       cf AS attempts,
       ROUND(100.0 * cf / NULLIF(cf + ca, 0), 1) AS attempt_share_pct,
       ROUND(100.0 * gf / NULLIF(sf, 0), 1) AS shooting_pct,
       pp_gf AS pp_goals,
       minors_drawn,
       sh_gf AS shorthanded_goals,
       pim,
       hits_for AS hits,
       blocks_for AS blocks,
       takeaways,
       giveaways,
       fo_wins AS faceoff_wins,
       ROUND(100.0 * fo_wins / NULLIF(fo_wins + fo_losses, 0), 1) AS faceoff_pct
FROM game_stats
ORDER BY is_home;